# Function Calling dan Tools: Solusi Latihan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/solusi/hari3/02-tools-function-calling-solusi.ipynb)

Ini **solusi lengkap** semua latihan bab ini (2 latihan) di Google Colab. Semua bagian `TODO` sudah terisi, jadi notebook ini tinggal dijalankan (**Runtime > Run all**, atau `Shift+Enter` per cell). Keluaran dari satu kali eksekusi dan penjelasan tiap latihan sudah disertakan; jawaban model berubah tiap kali dijalankan, jadi hasil Anda bisa sedikit berbeda. Kerjakan dulu [notebook latihannya](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/hari3/02-tools-function-calling-latihan.ipynb); buka solusi ini untuk mencocokkan hasil Anda atau saat macet.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-agent-migas.git"
REPO = "/content/workshop-agent-migas" if os.path.isdir("/content") else os.path.abspath("workshop-agent-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "main", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

## Bab 2 - Solusi Latihan: Tool Konversi Satuan dengan Siklus Eksekusi Lengkap
Jalankan sel satu per satu dengan Shift+Enter

Siklus 3 langkah: model merencanakan pemanggilan, kode menjalankan tool, model menjawab.

In [ ]:
# Langkah 1 - Menyiapkan model
import os
from typing import Literal

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Tool konversi satuan
Tabel faktor konversi, skema masukan, dan tool. Pasangan satuan yang tidak didukung ditolak dengan pesan yang jelas.

In [ ]:
FAKTOR = {
    ("bbl", "m3"): 0.158987, ("m3", "bbl"): 1 / 0.158987,
    ("psi", "bar"): 0.0689476, ("bar", "psi"): 14.5038,
    ("scf", "m3"): 0.0283168, ("m3", "scf"): 1 / 0.0283168,
}


class KonversiInput(BaseModel):
    nilai: float = Field(description="Angka yang dikonversi")
    dari: Literal["bbl", "m3", "psi", "bar", "scf"] = Field(description="Satuan asal")
    ke: Literal["bbl", "m3", "psi", "bar", "scf"] = Field(description="Satuan tujuan")


@tool(args_schema=KonversiInput)
def konversi_satuan(nilai: float, dari: str, ke: str) -> str:
    """Konversi satuan volume (bbl, m3, scf) atau tekanan (psi, bar). Pakai untuk setiap pertanyaan konversi satuan."""
    try:
        if dari == ke:
            return f"{nilai} {ke}"
        return f"{nilai} {dari} = {nilai * FAKTOR[(dari, ke)]:.3f} {ke}"
    except KeyError:
        return f"Konversi {dari} ke {ke} tidak didukung. Pasangan yang didukung: {sorted(set(FAKTOR))}"


model_dengan_tool = model.bind_tools([konversi_satuan])

## Langkah 3 - Fungsi siklus tiga langkah
`jalankan_siklus` menjalankan satu pertanyaan: model merencanakan, kode menjalankan setiap pemanggilan
(model bisa meminta lebih dari satu), lalu model menyusun jawaban akhir.

In [ ]:
def jalankan_siklus(model_dengan_tool, model, query: str):
    print(f"Pengguna: {query}")

    # Langkah 1: model merencanakan
    respons = model_dengan_tool.invoke(query)
    if not respons.tool_calls:
        print("  Model menjawab langsung tanpa tool:", respons.content, "\n")
        return

    messages = [HumanMessage(content=query), AIMessage(content=respons.content or "", tool_calls=respons.tool_calls)]

    # Langkah 2: kode menjalankan setiap pemanggilan (model bisa meminta lebih dari satu)
    for pemanggilan in respons.tool_calls:
        hasil = konversi_satuan.invoke(pemanggilan["args"])
        print(f"  Tool {pemanggilan['name']}({pemanggilan['args']}) -> {hasil}")
        messages.append(ToolMessage(content=hasil, tool_call_id=pemanggilan["id"]))

    # Langkah 3: model menyusun jawaban akhir
    print("  Jawaban akhir:", model.invoke(messages).content, "\n")

## Langkah 4 - Konversi tekanan

In [ ]:
jalankan_siklus(model_dengan_tool, model, "Tekanan wellhead SR-020 adalah 1500 psi. Berapa bar?")

Pengguna: Tekanan wellhead SR-020 adalah 1500 psi. Berapa bar?
  Tool konversi_satuan({'dari': 'psi', 'ke': 'bar', 'nilai': 1500}) -> 1500.0 psi = 103.421 bar
  Jawaban akhir: Tekanan wellhead SR-020 sebesar **1500 psi** setara dengan **103,4 bar**.

(Catatan: 1 psi ~ 0,0689476 bar) 



## Langkah 5 - Konversi volume

In [ ]:
jalankan_siklus(model_dengan_tool, model, "Tangki penampung berisi 2500 bbl. Berapa meter kubik?")

Pengguna: Tangki penampung berisi 2500 bbl. Berapa meter kubik?
  Tool konversi_satuan({'nilai': 2500, 'dari': 'bbl', 'ke': 'm3'}) -> 2500.0 bbl = 397.467 m3
  Jawaban akhir: Tangki penampung berisi **2500 barel (bbl)** setara dengan **397,467 meter kubik (m³)**.

**Perhitungan:**
1 barel (bbl) = 0,158987 m³  
Jadi, 2500 bbl × 0,158987 = **397,467 m³** 



## Langkah 6 - Dua konversi dalam satu pertanyaan

In [ ]:
jalankan_siklus(model_dengan_tool, model, "Berapa psi dalam 20 bar, dan berapa bbl dalam 10 m3?")

Pengguna: Berapa psi dalam 20 bar, dan berapa bbl dalam 10 m3?
  Tool konversi_satuan({'nilai': 20, 'dari': 'bar', 'ke': 'psi'}) -> 20.0 bar = 290.076 psi
  Tool konversi_satuan({'nilai': 10, 'dari': 'm3', 'ke': 'bbl'}) -> 10.0 m3 = 62.898 bbl
  Jawaban akhir: Berikut hasil konversinya:

**1. 20 bar ke psi**
20 bar sama dengan **290,08 psi** (dibulatkan dua desimal).

**2. 10 m³ ke bbl (barrel minyak)**
10 meter kubik sama dengan **62,90 bbl** (dibulatkan dua desimal).

*Catatan: Konversi bbl ini menggunakan standar **barrel minyak** (1 bbl = 158,9873 liter).*
*Jika yang dimaksud adalah **barrel cairan AS** (1 bbl = 119,24 liter) atau **barrel bir**, hasilnya akan berbeda.* 



## Langkah 7 - Pasangan satuan tidak valid
Tool harus menolak dengan sopan.

In [ ]:
jalankan_siklus(model_dengan_tool, model, "Ubah 100 bar menjadi bbl.")

Pengguna: Ubah 100 bar menjadi bbl.
  Model menjawab langsung tanpa tool: 

Maaf, konversi dari **bar** ke **bbl** tidak dapat dilakukan karena:

- **bar** adalah satuan **tekanan**
- **bbl** (barrel) adalah satuan **volume**

Keduanya mengukur besaran fisika yang berbeda, sehingga tidak bisa dikonversi secara langsung.

Konversi yang mungkin dilakukan:
- bar -> psi (sama-sama tekanan)
- bbl -> m³ atau scf (sama-sama volume, meskipun scf untuk gas)

Apakah Anda ingin mengonversi ke satuan yang sejenis? 



### Penjelasan Solusi Latihan 2.1: siklus function calling

Model tidak pernah menjalankan tool. Ia hanya meminta, dan kode Anda yang menjalankan lalu melaporkan hasilnya. Fungsi `jalankan_siklus` menulis tiga langkah itu secara eksplisit.

- `TODO(B2.1)`: skema `KonversiInput` memakai `Literal` untuk satuan, sehingga model tidak bisa meminta satuan di luar daftar.
- `TODO(B2.2)`: isi tool. Pasangan satuan yang tidak ada di `FAKTOR` menghasilkan `KeyError`, yang diubah menjadi pesan yang menyebut pasangan yang didukung.
- `TODO(B2.3)`: `bind_tools` memberi tahu model tool apa yang tersedia.
- `TODO(B2.4)`: kode menjalankan tool dengan argumen dari `pemanggilan["args"]`.
- `TODO(B2.5)`: hasilnya dikirim kembali sebagai `ToolMessage` dengan `tool_call_id` yang sama. ID itu yang mengikat jawaban ke permintaan; tanpa itu penyedia menolak riwayatnya.

Pertanyaan dua konversi sekaligus (bar ke psi dan m3 ke bbl) menghasilkan dua pemanggilan dalam satu respons, sehingga loop `for pemanggilan in respons.tool_calls` memang perlu. Satu hal yang jujur perlu dicatat: pada langkah terakhir ("100 bar menjadi bbl") model di eksekusi ini menjawab langsung bahwa tekanan dan volume tidak bisa dikonversi, tanpa memanggil tool, jadi pesan penolakan dari tool tidak terlihat. Hasilnya benar, tetapi bukan jalur yang ingin diuji. Model lain bisa saja memanggil tool dan menerima pesan "tidak didukung".

## Bab 2 - Solusi Latihan Bonus: Asisten Pemeriksaan Sumur dengan Banyak Tool
Jalankan sel satu per satu dengan Shift+Enter

Model memilih sendiri tool yang tepat; kode menjalankannya lewat peta nama -> tool.

In [ ]:
# Langkah 1 - Menyiapkan model
import os

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

from migas_shared.field_tools import TOOL_LAPANGAN

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Menambah tool estimasi kehilangan produksi
Tool lapangan dari paket bersama digabung dengan satu tool baru, lalu dipetakan nama -> tool.

In [ ]:
class KehilanganInput(BaseModel):
    minyak_bopd: float = Field(ge=0, description="Produksi minyak normal sumur dalam bopd")
    hari_henti: float = Field(ge=0, description="Lama sumur berhenti dalam hari")


@tool(args_schema=KehilanganInput)
def estimasi_kehilangan_produksi(minyak_bopd: float, hari_henti: float) -> str:
    """Perkirakan produksi minyak yang hilang (deferred production) selama sumur berhenti: bopd dikali jumlah hari."""
    return f"Estimasi kehilangan produksi: {minyak_bopd * hari_henti:,.0f} bbl selama {hari_henti} hari"


SEMUA_TOOL = [*TOOL_LAPANGAN, estimasi_kehilangan_produksi]
PETA_TOOL = {t.name: t for t in SEMUA_TOOL}

## Langkah 3 - Loop pemanggilan tool
Loop sederhana: model meminta tool, kode menjalankan, ulangi sampai model menjawab dengan teks.

In [ ]:
def jawab(model, pertanyaan: str, maks_putaran: int = 4) -> str:
    """Loop sederhana: model meminta tool, kode menjalankan, ulangi sampai model menjawab dengan teks."""
    model_dengan_tool = model.bind_tools(SEMUA_TOOL)
    messages = [
        SystemMessage(content="Anda asisten pemeriksaan sumur di GS-SR1. Pakai tool bila perlu, jawab ringkas dalam Bahasa Indonesia."),
        HumanMessage(content=pertanyaan),
    ]
    for _ in range(maks_putaran):
        respons = model_dengan_tool.invoke(messages)
        if not respons.tool_calls:
            return str(respons.content)
        messages.append(AIMessage(content=respons.content or "", tool_calls=respons.tool_calls))
        for pemanggilan in respons.tool_calls:
            try:
                hasil = PETA_TOOL[pemanggilan["name"]].invoke(pemanggilan["args"])
            except Exception as error:  # tool yang gagal dilaporkan ke model, bukan menghentikan program
                hasil = f"Error: {error}"
            print(f"   [tool] {pemanggilan['name']}({pemanggilan['args']}) -> {hasil}")
            messages.append(ToolMessage(content=str(hasil), tool_call_id=pemanggilan["id"]))
    return "Batas putaran tercapai tanpa jawaban akhir."

## Langkah 4 - Status sumur dan water cut

In [ ]:
pertanyaan = "Bagaimana kondisi sumur SR-020, dan berapa water cut-nya?"
print(f"Pengguna: {pertanyaan}")
print("Asisten :", jawab(model, pertanyaan), "\n")

Pengguna: Bagaimana kondisi sumur SR-020, dan berapa water cut-nya?
   [tool] cek_status_sumur({'kode_sumur': 'SR-020'}) -> Sumur SR-020 (Seruni, ESP): status Produksi, minyak 120 bopd, air 480 bwpd. Catatan: ESP trip overload kemarin sore, stabil sejak pukul 18.00
   [tool] hitung_water_cut({'oil_bopd': 120, 'water_bwpd': 480}) -> Water cut = 80.0% (total cairan 600 bpd)
Asisten : **Kondisi Sumur SR-020 (Seruni, GS-SR1):**
- **Area:** Seruni
- **Jenis Pompa:** ESP (Electric Submersible Pump)
- **Status:** Produksi
- **Produksi Minyak:** 120 bopd
- **Produksi Air:** 480 bwpd
- **Catatan:** ESP trip *overload* kemarin sore, namun sudah stabil sejak pukul 18.00.

**Water Cut:** **80%** (dari total cairan 600 bpd).

Kesimpulan: Sumur berproduksi normal dengan water cut yang tinggi (80%). Perlu diwaspadai potensi *trip* ESP berulang akibat overload. 



## Langkah 5 - Estimasi kehilangan produksi

In [ ]:
pertanyaan = "Sumur KN-033 menunggu pulling selama 12 hari. Jika produksi normalnya 45 bopd, berapa produksi yang hilang?"
print(f"Pengguna: {pertanyaan}")
print("Asisten :", jawab(model, pertanyaan), "\n")

Pengguna: Sumur KN-033 menunggu pulling selama 12 hari. Jika produksi normalnya 45 bopd, berapa produksi yang hilang?
   [tool] estimasi_kehilangan_produksi({'hari_henti': 12, 'minyak_bopd': 45}) -> Estimasi kehilangan produksi: 540 bbl selama 12.0 hari
Asisten : Produksi yang hilang (deferred production) untuk sumur KN-033 selama 12 hari dengan produksi normal 45 bopd adalah **540 barel (bbl)**. 



## Langkah 6 - Konversi tekanan

In [ ]:
pertanyaan = "Tekanan kepala sumur ML-012 tercatat 900 psi. Berapa bar?"
print(f"Pengguna: {pertanyaan}")
print("Asisten :", jawab(model, pertanyaan), "\n")

Pengguna: Tekanan kepala sumur ML-012 tercatat 900 psi. Berapa bar?
   [tool] konversi_satuan({'nilai': 900, 'dari': 'psi', 'ke': 'bar'}) -> 900.0 psi = 62.053 bar
Asisten : 900 psi setara dengan **62,053 bar**. 



### Penjelasan Solusi Latihan 2.2: asisten dengan banyak tool

Bedanya dengan latihan sebelumnya: model memilih sendiri tool mana yang dipakai dan boleh meminta beberapa putaran, jadi siklusnya menjadi loop.

- `TODO(B2.6)`: tool baru `estimasi_kehilangan_produksi` ditambahkan ke tool lapangan dari paket bersama. Skema `ge=0` menolak angka negatif sebelum tool berjalan.
- `TODO(B2.7)`: tiap putaran memanggil model dengan seluruh `messages`.
- `TODO(B2.8)`: permintaan model disimpan sebagai `AIMessage` yang membawa `tool_calls`, supaya hasil tool yang menyusul punya pasangan.
- `TODO(B2.9)`: `PETA_TOOL` memetakan nama ke tool, jadi satu loop melayani semua tool. Kegagalan tool ditangkap dan dikirim ke model sebagai teks "Error: ...", sehingga model bisa memperbaiki permintaannya alih-alih program berhenti.
- `TODO(B2.10)`: hasil dikirim kembali sebagai `ToolMessage`.

Pada keluaran, pertanyaan pertama memicu dua tool berurutan (status sumur lalu hitung water cut dari angka yang baru diterima), pertanyaan kedua hanya satu tool (45 bopd kali 12 hari menjadi 540 bbl). `maks_putaran=4` adalah pengaman: model yang terus meminta tool tidak akan membuat loop berjalan selamanya.